# Data Quality Cleaning Notebook

This notebook uses Gemini 3.1 Flash lite to perform advanced data quality cleaning on the dataset. It determines if a category should be preserved and whether individual texts should be kept, removed, or edited to remove hints.

In [ ]:
!pip install -q google-genai pandas

import os
import json
import time
from google import genai
from datetime import datetime
from google.colab import userdata

# Retrieve the secret from Colab's secret manager
try:
    api_key = userdata.get('GOOGLE_API_KEY')
    os.environ['GOOGLE_API_KEY'] = api_key
except Exception as e:
    print(f"Error accessing API key: {e}")

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab or drive mount failed.')

# Configuration
CONFIG = {
    'MODEL': 'gemini-3.1-flash-lite-latest',
    'INPUT_PATH': '/content/drive/MyDrive/Graphiko/exports/base_data/latest/channel_titles.json',
    'OUTPUT_PATH': '/content/drive/MyDrive/Graphiko/exports/base_data/latest/channel_titles_cleaned.json',
    'CACHE_DIR': '/content/drive/MyDrive/classification_outputs',
    'REUSE_CACHE': True,
    'API_KEY': os.getenv('GOOGLE_API_KEY')
}

if not os.path.exists(CONFIG['CACHE_DIR']):
    os.makedirs(CONFIG['CACHE_DIR'], exist_ok=True)

client = genai.Client(api_key=CONFIG['API_KEY'])

In [ ]:
def get_cache_path(name, timestamp=None):
    if timestamp:
        return os.path.join(CONFIG['CACHE_DIR'], f'{name}_{timestamp}.json')
    return os.path.join(CONFIG['CACHE_DIR'], f'{name}_latest.json')

def save_cache(name, data):
    latest_path = get_cache_path(name)
    with open(latest_path, 'w') as f:
        json.dump(data, f, indent=2)

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    versioned_path = get_cache_path(name, timestamp=timestamp)
    with open(versioned_path, 'w') as f:
        json.dump(data, f, indent=2)

def load_cache(name):
    path = get_cache_path(name)
    if CONFIG['REUSE_CACHE'] and os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    return None

In [ ]:
def clean_dataset(input_path):
    if not os.path.exists(input_path):
        print(f'Error: Input file not found at {input_path}')
        return

    with open(input_path, 'r') as f:
        data = json.load(f)

    cleaning_results = load_cache('cleaning_results') or {}
    
    for category, texts in data.items():
        if category in cleaning_results:
            print(f'Using cached results for category: {category}')
            continue

        print(f'Processing category: {category}...')
        
        prompt = f"""
        Evaluate the following category and its text samples for data quality, specifically looking for 'hints' where the category name or parts of it appear in the text.
        
        Category: {category}
        Texts:
        {json.dumps(texts, indent=2)}
        
        Task:
        1. Should this category be used or skipped? (Skip if it's inherently biased or unsuitable for unbiased classification evaluation).
        2. For each text, determine if it should be kept as is, removed (if it contains strong hints that cannot be edited out), or edited (to remove hints while preserving semantic meaning).
        
        Return a JSON object with this structure:
        {{
          "decision": "use" | "skip",
          "reason": "explanation",
          "results": [
            {{"original": "original text", "status": "keep" | "remove" | "edit", "edited": "edited text if status is edit, else null"}}
          ]
        }}
        """
        
        try:
            response = client.models.generate_content(model=CONFIG['MODEL'], contents=prompt)
            res_text = response.text.strip()
            if '```json' in res_text: res_text = res_text.split('```json')[1].split('```')[0]
            
            result = json.loads(res_text)
            cleaning_results[category] = result
            
            # Print Summary
            print(f"Category: {category} -> Decision: {result['decision'].upper()}")
            if result['decision'] == 'use':
                removed = [r for r in result['results'] if r['status'] == 'remove']
                edited = [r for r in result['results'] if r['status'] == 'edit']
                print(f"  - Removed: {len(removed)}, Edited: {len(edited)}")
                if removed:
                    print(f"  - Example Removed: {removed[0]['original']}")
                if edited:
                    print(f"  - Example Edited: {edited[0]['original']} -> {edited[0]['edited']}")
            
            save_cache('cleaning_results', cleaning_results)
            time.sleep(2) # Avoid throttling
        except Exception as e:
            print(f'Error processing category {category}: {e}')
            time.sleep(5)

    return cleaning_results

results = clean_dataset(CONFIG['INPUT_PATH'])

In [ ]:
def finalize_and_stats(cleaning_results):
    cleaned_data = {}
    categories_preserved = 0
    categories_skipped = 0
    texts_removed = 0
    texts_modified = 0
    total_texts_original = 0
    total_texts_final = 0

    for category, info in cleaning_results.items():
        if info['decision'] == 'skip':
            categories_skipped += 1
            continue
        
        categories_preserved += 1
        category_texts = []
        for res in info['results']:
            total_texts_original += 1
            if res['status'] == 'keep':
                category_texts.append(res['original'])
            elif res['status'] == 'edit':
                category_texts.append(res['edited'])
                texts_modified += 1
            elif res['status'] == 'remove':
                texts_removed += 1
        
        if category_texts:
            cleaned_data[category] = category_texts
            total_texts_final += len(category_texts)

    # Save cleaned dataset
    with open(CONFIG['OUTPUT_PATH'], 'w') as f:
        json.dump(cleaned_data, f, indent=2)
    
    print("\n--- Dataset Level Statistics ---")
    print(f"Categories Preserved: {categories_preserved}")
    print(f"Categories Skipped: {categories_skipped}")
    print(f"Total Texts Original: {total_texts_original}")
    print(f"Texts Removed: {texts_removed}")
    print(f"Texts Modified: {texts_modified}")
    print(f"Total Texts in Cleaned Dataset: {total_texts_final}")
    print(f"Cleaned dataset saved to: {CONFIG['OUTPUT_PATH']}")

if results:
    finalize_and_stats(results)